# Kalvium Brain 5.34: Handling Missing Values (Drop vs Fill)

This notebook demonstrates responsible handling of missing values in Pandas.

## Learning Goals
- Detect missing values and quantify missingness
- Apply row and column drop strategies intentionally
- Fill missing values using constants, median/mean, and mode
- Compare trade-offs between dropping and filling
- Validate cleaned results and avoid common mistakes

## 1) Import Pandas and Load Dataset with Missing Values

We load the CSV and create a working copy so the original DataFrame stays unchanged.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Resolve the dataset path robustly from the notebook location.
possible_paths = [
    Path('employee-survey-analysis/data/raw/Employee Attrition.csv'),
    Path('./employee-survey-analysis/data/raw/Employee Attrition.csv')
]

csv_path = None
for p in possible_paths:
    if p.exists():
        csv_path = p
        break

if csv_path is None:
    raise FileNotFoundError('Could not find Employee Attrition.csv in expected location.')

df_original = pd.read_csv(csv_path)
df = df_original.copy()

print(f'Dataset loaded from: {csv_path}')
print(f'Original shape: {df.shape}')
df.head()

Dataset loaded from: employee-survey-analysis\data\raw\Employee Attrition.csv
Original shape: (15787, 10)


,Emp ID,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,promotion_last_5years,dept,salary
0,1.0,0.38,0.53,2.0,157.0,3.0,0.0,0.0,sales,low
1,2.0,0.80,0.86,5.0,262.0,6.0,0.0,0.0,sales,medium
2,3.0,0.11,0.88,7.0,272.0,4.0,0.0,0.0,sales,medium
3,4.0,0.72,0.87,5.0,223.0,5.0,0.0,0.0,sales,low
4,5.0,0.37,0.52,2.0,159.0,3.0,0.0,0.0,sales,low


## 2) Inspect Missing Values and Baseline Shape

We inspect overall shape, schema, missing counts, and missing percentages.

If the dataset has very few missing values, we create a demo copy with synthetic missing values so all strategies can be demonstrated clearly.

In [2]:
print('Baseline shape:', df.shape)
print('\nDataFrame info:')
df.info()

missing_count = df.isna().sum().sort_values(ascending=False)
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)

missing_summary = pd.DataFrame({
    'missing_count': missing_count,
    'missing_pct': missing_pct.round(2)
})

print('\nColumns with missing values:')
display(missing_summary[missing_summary['missing_count'] > 0])
print('Total missing cells in original data:', int(df.isna().sum().sum()))

# Work on a demo copy to ensure strategies can be shown even if original data has limited nulls.
df_work = df.copy()

if df_work.isna().sum().sum() < 20:
    rng = np.random.default_rng(42)

    numeric_cols = df_work.select_dtypes(include='number').columns.tolist()
    categorical_cols = df_work.select_dtypes(exclude='number').columns.tolist()

    n_rows = len(df_work)

    if numeric_cols:
        sample_rows_num = rng.choice(n_rows, size=max(5, n_rows // 20), replace=False)
        df_work.loc[sample_rows_num, numeric_cols[0]] = np.nan

    if len(numeric_cols) > 1:
        sample_rows_num2 = rng.choice(n_rows, size=max(5, n_rows // 25), replace=False)
        df_work.loc[sample_rows_num2, numeric_cols[1]] = np.nan

    if categorical_cols:
        sample_rows_cat = rng.choice(n_rows, size=max(5, n_rows // 20), replace=False)
        df_work.loc[sample_rows_cat, categorical_cols[0]] = np.nan

print('\nWorking copy shape:', df_work.shape)
print('Total missing cells in working copy:', int(df_work.isna().sum().sum()))

Baseline shape: (15787, 10)

DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15787 entries, 0 to 15786
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Emp ID                 14999 non-null  float64
 1   satisfaction_level     14999 non-null  float64
 2   last_evaluation        14999 non-null  float64
 3   number_project         14999 non-null  float64
 4   average_montly_hours   14999 non-null  float64
 5   time_spend_company     14999 non-null  float64
 6   Work_accident          14999 non-null  float64
 7   promotion_last_5years  14999 non-null  float64
 8   dept                   14999 non-null  object 
 9   salary                 14999 non-null  object 
dtypes: float64(8), object(2)
memory usage: 1.2+ MB

Columns with missing values:


,missing_count,missing_pct
Emp ID,788,4.99
satisfaction_level,788,4.99
last_evaluation,788,4.99
number_project,788,4.99
average_montly_hours,788,4.99
time_spend_company,788,4.99
Work_accident,788,4.99
promotion_last_5years,788,4.99
dept,788,4.99
salary,788,4.99


Total missing cells in original data: 7880

Working copy shape: (15787, 10)
Total missing cells in working copy: 7880


## 3) Drop Rows with Missing Values (`dropna`)

This removes every row that has at least one missing value. It is simple, but can reduce dataset size significantly.

In [3]:
before_rows_shape = df_work.shape
df_drop_rows = df_work.dropna()

print('Before drop rows shape:', before_rows_shape)
print('After drop rows shape :', df_drop_rows.shape)
print('Rows removed          :', before_rows_shape[0] - df_drop_rows.shape[0])
print('Remaining missing cells:', int(df_drop_rows.isna().sum().sum()))

Before drop rows shape: (15787, 10)
After drop rows shape : (14999, 10)
Rows removed          : 788
Remaining missing cells: 0


## 4) Drop Columns with Excessive Missingness

We set a threshold and drop only columns with too many nulls.

Threshold used here: columns with more than 40% missing values.

In [4]:
missing_fraction = df_work.isna().mean()
cols_to_drop = missing_fraction[missing_fraction > 0.40].index.tolist()

df_drop_cols = df_work.drop(columns=cols_to_drop)

print('Columns selected for drop (>40% missing):', cols_to_drop if cols_to_drop else 'None')
print('Shape before dropping columns:', df_work.shape)
print('Shape after dropping columns :', df_drop_cols.shape)
print('Remaining missing cells      :', int(df_drop_cols.isna().sum().sum()))

Columns selected for drop (>40% missing): None
Shape before dropping columns: (15787, 10)
Shape after dropping columns : (15787, 10)
Remaining missing cells      : 7880


## 5) Fill Missing Values with Constants (`fillna`)

Use explicit constants where domain-appropriate:
- Categorical columns: `"Unknown"`
- Numeric columns: `0` only when that value is meaningful

In [5]:
df_fill_constant = df_work.copy()

num_cols = df_fill_constant.select_dtypes(include='number').columns.tolist()
cat_cols = df_fill_constant.select_dtypes(exclude='number').columns.tolist()

# Demonstration-only: fill categorical with a safe label.
if cat_cols:
    df_fill_constant[cat_cols] = df_fill_constant[cat_cols].fillna('Unknown')

# Fill numeric with 0 ONLY as a demonstration of constant filling.
if num_cols:
    df_fill_constant[num_cols] = df_fill_constant[num_cols].fillna(0)

print('Shape after constant fill:', df_fill_constant.shape)
print('Remaining missing cells  :', int(df_fill_constant.isna().sum().sum()))

Shape after constant fill: (15787, 10)
Remaining missing cells  : 0


## 6) Fill Numeric Missing Values with Mean/Median

Mean works for fairly symmetric distributions; median is often safer when values are skewed or contain outliers.

In [6]:
df_fill_numeric = df_work.copy()
num_cols = df_fill_numeric.select_dtypes(include='number').columns.tolist()

for col in num_cols:
    if df_fill_numeric[col].isna().any():
        median_value = df_fill_numeric[col].median()
        df_fill_numeric[col] = df_fill_numeric[col].fillna(median_value)

print('Numeric columns imputed with median where needed.')
print('Remaining missing cells after numeric fill:', int(df_fill_numeric.isna().sum().sum()))

Numeric columns imputed with median where needed.
Remaining missing cells after numeric fill: 1576


## 7) Fill Categorical Missing Values with Mode

For each categorical column, fill with mode (most frequent value). If mode is unavailable, use `"Unknown"`.

In [7]:
df_fill_mode = df_fill_numeric.copy()
cat_cols = df_fill_mode.select_dtypes(exclude='number').columns.tolist()

for col in cat_cols:
    if df_fill_mode[col].isna().any():
        mode_series = df_fill_mode[col].mode(dropna=True)
        fill_value = mode_series.iloc[0] if not mode_series.empty else 'Unknown'
        df_fill_mode[col] = df_fill_mode[col].fillna(fill_value)

print('Categorical columns imputed with mode/Unknown.')
print('Final missing cells after numeric+categorical fill:', int(df_fill_mode.isna().sum().sum()))

Categorical columns imputed with mode/Unknown.
Final missing cells after numeric+categorical fill: 0


## 8) Compare Drop vs Fill Outcomes (Shape and Null Counts)

We compare three versions:
- Working copy (before handling)
- Drop strategy (`dropna` rows)
- Fill strategy (median + mode)

This helps choose the least harmful strategy for the use case.

In [8]:
def frame_stats(name, dataf):
    return {
        'version': name,
        'rows': dataf.shape[0],
        'columns': dataf.shape[1],
        'missing_cells': int(dataf.isna().sum().sum())
    }

comparison = pd.DataFrame([
    frame_stats('working_copy', df_work),
    frame_stats('drop_rows', df_drop_rows),
    frame_stats('fill_median_mode', df_fill_mode)
])

display(comparison)

,version,rows,columns,missing_cells
0,working_copy,15787,10,7880
1,drop_rows,14999,10,0
2,fill_median_mode,15787,10,0


## 9) Validate Cleaned Data and Guard Against Common Mistakes

Checks performed:
- Verify remaining nulls
- Ensure categorical columns were not filled with numeric values
- Confirm critical columns still exist (not dropped blindly)

In [9]:
critical_columns = ['Attrition'] if 'Attrition' in df_work.columns else []

print('Missing cells in drop strategy :', int(df_drop_rows.isna().sum().sum()))
print('Missing cells in fill strategy :', int(df_fill_mode.isna().sum().sum()))

if critical_columns:
    dropped_critical = [c for c in critical_columns if c not in df_drop_cols.columns]
    print('Critical columns removed by column-drop strategy:', dropped_critical if dropped_critical else 'None')

cat_cols_final = df_fill_mode.select_dtypes(exclude='number').columns
wrong_fill_flags = []
for col in cat_cols_final:
    # If any non-null value is numeric after fill, flag it for review.
    non_null_values = df_fill_mode[col].dropna()
    if len(non_null_values) > 0 and pd.api.types.is_numeric_dtype(non_null_values):
        wrong_fill_flags.append(col)

print('Categorical columns with suspicious numeric-only values:', wrong_fill_flags if wrong_fill_flags else 'None detected')

Missing cells in drop strategy : 0
Missing cells in fill strategy : 0
Categorical columns with suspicious numeric-only values: None detected


## 10) Export Cleaned Variants for Submission and Video Walkthrough

We save both strategies for clear evidence in your PR/video.

### Walkthrough checklist (~2 mins)
- Show baseline shape and missing summary
- Show `dropna` result and shape loss
- Show fill strategy (constant + median/mode)
- Show comparison table
- Explain why final strategy is reasonable

In [10]:
output_dir = Path('employee-survey-analysis/outputs/processed')
output_dir.mkdir(parents=True, exist_ok=True)

drop_output_path = output_dir / 'drop_strategy.csv'
fill_output_path = output_dir / 'fill_strategy.csv'

df_drop_rows.to_csv(drop_output_path, index=False)
df_fill_mode.to_csv(fill_output_path, index=False)

print(f'Saved: {drop_output_path}')
print(f'Saved: {fill_output_path}')

Saved: employee-survey-analysis\outputs\processed\drop_strategy.csv
Saved: employee-survey-analysis\outputs\processed\fill_strategy.csv
